# Machine Learning: Feature Engineering, Model Training & Evaluation
In this notebook, we prepare features, train classification models, evaluate their performance, and save the best model for deployment.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
from xgboost import XGBClassifier

# Optional: imblearn for SMOTE if installed
# from imblearn.over_sampling import SMOTE

## 1. Load Cleaned Data

In [2]:
df = pd.read_csv('../data/processed/stroke-data-cleaned.csv')
df.head()

,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,28.1,never smoked,1
2,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1


## 2. Feature Engineering & Preprocessing Setup

In [3]:
X = df.drop('stroke', axis=1)
y = df['stroke']

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training set shape: {X_train.shape}")
print(f"Test set shape: {X_test.shape}")

Training set shape: (4087, 10)
Test set shape: (1022, 10)


In [4]:
# Define categorical and numerical columns
categorical_cols = ['gender', 'ever_married', 'work_type', 'Residence_type', 'smoking_status']
numerical_cols = ['age', 'avg_glucose_level', 'bmi', 'hypertension', 'heart_disease']

# Create preprocessing steps
numeric_transformer = StandardScaler()
categorical_transformer = OneHotEncoder(handle_unknown='ignore')

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numerical_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])

## 3. Model Training & Evaluation (Random Forest)

In [5]:
# Create a pipeline with preprocessor and Random Forest
rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42))
])

# Train the model
rf_pipeline.fit(X_train, y_train)

# Predict and evaluate
y_pred_rf = rf_pipeline.predict(X_test)
print("Random Forest Classification Report:")
print(classification_report(y_test, y_pred_rf))
print(f"ROC-AUC Score: {roc_auc_score(y_test, rf_pipeline.predict_proba(X_test)[:, 1]):.4f}")

Random Forest Classification Report:
              precision    recall  f1-score   support

           0       0.95      1.00      0.97       972
           1       0.00      0.00      0.00        50

    accuracy                           0.95      1022
   macro avg       0.48      0.50      0.49      1022
weighted avg       0.90      0.95      0.93      1022

ROC-AUC Score: 0.7832


## 4. Model Training & Evaluation (XGBoost)

In [6]:
# Create a pipeline with XGBoost
xgb_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', XGBClassifier(scale_pos_weight=len(y_train[y_train==0])/len(y_train[y_train==1]), random_state=42))
])

# Train
xgb_pipeline.fit(X_train, y_train)

# Predict and evaluate
y_pred_xgb = xgb_pipeline.predict(X_test)
print("XGBoost Classification Report:")
print(classification_report(y_test, y_pred_xgb))
print(f"ROC-AUC Score: {roc_auc_score(y_test, xgb_pipeline.predict_proba(X_test)[:, 1]):.4f}")

XGBoost Classification Report:


              precision    recall  f1-score   support

           0       0.96      0.95      0.96       972
           1       0.19      0.22      0.20        50

    accuracy                           0.92      1022
   macro avg       0.57      0.59      0.58      1022
weighted avg       0.92      0.92      0.92      1022

ROC-AUC Score: 0.7612


## 5. Save the Best Model

In [7]:
# XGBoost usually performs well, or we can save Random Forest. Let's save Random Forest for this example.
# (In practice, choose the one with better metrics based on your goals).
model_path = '../models/stroke_prediction_pipeline.pkl'
joblib.dump(rf_pipeline, model_path)
print(f"Model pipeline saved to {model_path}")

Model pipeline saved to ../models/stroke_prediction_pipeline.pkl
